<a href="https://colab.research.google.com/github/lucasjamiro/Experiment-PPGCG/blob/main/%E2%80%8EGA155%20-%20SQL%20magic%20bairros%202.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# install
!apt install postgresql postgresql-contrib &>log
!apt install gdal-bin
!service postgresql start
!sudo -u postgres psql -c "CREATE USER root WITH SUPERUSER"
# set connection
%load_ext sql
%config SqlMagic.feedback=False
%config SqlMagic.autopandas=True
%sql postgresql+psycopg2://@/postgres
!apt install postgis
# Setup a password `postgres` for username `postgres`
!sudo -u postgres psql -U postgres -c "ALTER USER postgres PASSWORD 'postgres';"

In [ ]:
!pip install --upgrade ipython-sql prettytable


In [ ]:
import prettytable

In [ ]:
%sql CREATE EXTENSION postgis

In [ ]:
# Preparando o ambiente no Google Drive:

#importando a biblioteca
from google.colab import drive

# Isso irá pedir sua autorização
drive.mount('/content/drive',force_remount=True)

# Agora, seu Drive estará disponível em: /content/drive/My Drive

In [ ]:
!ogr2ogr -f PostgreSQL PG:"host=localhost user=postgres password=postgres dbname=postgres" -s_srs EPSG:4326 -t_srs EPSG:4326 -nln public.bairros1 -lco GEOMETRY_NAME=geom -overwrite /content/drive/MyDrive/Dados/bairros_wgs84.geojson

In [ ]:
df_bairros = %sql SELECT * FROM bairros1
df_bairros.head()

In [ ]:
%sql CREATE OR REPLACE VIEW bairro_selecionado AS SELECT *, ST_AsText(geom) as geometria from bairros1 WHERE nome = 'Crisro rei'
df = %sql SELECT * from bairro_selecionado
df.head()

In [ ]:
df1 = %sql SELECT bairros1.*, ST_AsText(bairros1.geom) as geometria from bairros1, bairro_selecionado WHERE ST_touches(bairro_selecionado.geom,bairros1.geom) = TRUE
df1

In [ ]:
bairroin = input("Digite o nome do bairro: ")

%sql SET SESSION search_path TO public
df = %sql SELECT shape_area FROM bairros1 WHERE LOWER(nome) = LOWER(:bairroin)

if not df.empty:
    df['shape_area'] = df['shape_area'] / 1000000
    print(f"Área do bairro {bairroin}: {df['shape_area'].iloc[0]:.2f} km²")
else:
    print(f"O bairro '{bairroin}' não foi encontrado. Por favor, verifique o nome (ex: 'SEMINÁRIO').")

df.head()

In [ ]:
from ipywidgets import Dropdown
from IPython.display import display, clear_output
import pandas as pd

# 1. Buscar bairros com SQLMagic
# O resultado já é um dataframe pandas por padrão
bairro_select = %sql SELECT codigo, nome FROM bairros1 ORDER BY nome

# 2. Criar Dropdown com os bairros
comboMun = Dropdown(
    options=[(row['nome'], row['codigo']) for _, row in bairro_select.iterrows()],
    description='Bairro:'
)

# 3. Definir função de callback
def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        clear_output(wait=True)
        display(comboMun)

        # Executa consulta com o código selecionado e acha o nome
        selected_codigo = comboMun.value
        selected_bairro_name = bairro_select[bairro_select['codigo'] == selected_codigo]['nome'].iloc[0]

        %sql SET SESSION search_path TO public
        df = %sql SELECT shape_area FROM bairros1 WHERE LOWER(nome) = LOWER(:selected_bairro_name)

        if not df.empty:
            df['shape_area'] = df['shape_area'] / 1000000
            print(f"Área do bairro {selected_bairro_name}: {df['shape_area'].iloc[0]:.2f} km²")
        else:
            # Idealmente essa função não deve ser ativada nunca
            print(f"O bairro '{selected_bairro_name}' não foi encontrado. Por favor, verifique o nome.")

# 4. Associar a função ao Dropdown
comboMun.observe(on_change)

# 5. Mostrar o Dropdown na interface
display(comboMun)


In [ ]:
from ipywidgets import Dropdown
from IPython.display import display, clear_output
import pandas as pd
import geopandas as gpd
import folium
from shapely import wkt

# 1. Buscar bairros
df_bairros = %sql SELECT codigo, nome FROM bairros1 ORDER BY nome

# 2. Criar Dropdown
comboMun = Dropdown(
    options=[(row['nome'], row['codigo']) for _, row in df_bairros.iterrows()],
    description='Bairros:'
)

# 3. Callback ao mudar a seleção
def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        clear_output(wait=True)
        display(comboMun)

        codigo = comboMun.value

        # Consulta área
        # Executa consulta com o código selecionado e acha o nome
        selected_codigo = comboMun.value
        selected_bairro_name = bairro_select[bairro_select['codigo'] == selected_codigo]['nome'].iloc[0]

        %sql SET SESSION search_path TO public
        df = %sql SELECT shape_area FROM bairros1 WHERE LOWER(nome) = LOWER(:selected_bairro_name)

        if not df.empty:
            df['shape_area'] = df['shape_area'] / 1000000
            print(f"Área do bairro {selected_bairro_name}: {df['shape_area'].iloc[0]:.2f} km²")
        else:
            # Idealmente essa função não deve ser ativada nunca
            print(f"O bairro '{selected_bairro_name}' não foi encontrado. Por favor, verifique o nome.")

        # Consulta geometria como WKT para exibir no mapa
        consulta_geom = f"""
            SELECT ST_AsText(ST_Transform(geom, 4326)) AS geom
            FROM bairros1
            WHERE codigo = '{codigo}'
        """
        resultado_geom = %sql $consulta_geom
        resultado_geom['geom'] = resultado_geom['geom'].apply(wkt.loads)
        gdf = gpd.GeoDataFrame(resultado_geom, geometry="geom", crs="EPSG:4326")

        # Criar o mapa
        centro = gdf.geometry.centroid.iloc[0]
        mapa = folium.Map(location=[centro.y, centro.x], zoom_start=10)
        folium.GeoJson(gdf).add_to(mapa)

        display(mapa)

# 4. Associar o callback
comboMun.observe(on_change)

# 5. Exibir a interface
display(comboMun)

In [ ]:
from ipywidgets import Dropdown
from IPython.display import display, clear_output
import pandas as pd
import geopandas as gpd
import folium
from shapely import wkt

# 1. Buscar bairros
df_bairros = %sql SELECT codigo, nome FROM bairros1 ORDER BY nome

# 2. Criar Dropdown
comboMun = Dropdown(
    options=[(row['nome'], row['codigo']) for _, row in df_bairros.iterrows()],
    description='Bairros:'
)

# 3. Função para atualizar o mapa
def atualizar_mapa(change):
    if change['type'] == 'change' and change['name'] == 'value':
        clear_output(wait=True)
        display(comboMun)

        codigo = comboMun.value

        # Consulta bairro selecionado (original shape)
        consulta_sel = f"""
            SELECT codigo, nome, ST_AsText(ST_Transform(geom, 4326)) AS geom
            FROM bairros1
            WHERE codigo = '{codigo}'
        """
        df_sel = %sql $consulta_sel
        df_sel['geometry'] = df_sel['geom'].apply(wkt.loads)
        gdf_sel = gpd.GeoDataFrame(df_sel, geometry='geometry', crs='EPSG:4326')

        # Consulta vizinhos, usando o buffer do bairro selecionado para a detecção de 'touches'
        consulta_viz = f"""
            SELECT codigo, nome, ST_AsText(ST_Transform(geom, 4326)) AS geom
            FROM bairros1
            WHERE codigo != '{codigo}'
              AND ST_Intersects(
                    (SELECT ST_Buffer(ST_Transform(geom, 3857), 10) FROM bairros1 WHERE codigo = '{codigo}'),
                    ST_Transform(bairros1.geom, 3857)
                  )
        """
        df_viz = %sql $consulta_viz

        # Adiciona verificação para df_viz vazio
        if not df_viz.empty:
            df_viz['geometry'] = df_viz['geom'].apply(wkt.loads)
            gdf_viz = gpd.GeoDataFrame(df_viz, geometry='geometry', crs='EPSG:4326')
        else:
            # Cria um GeoDataFrame vazio se não houver vizinhos
            gdf_viz = gpd.GeoDataFrame(columns=['codigo', 'nome', 'geom', 'geometry'], geometry='geometry', crs='EPSG:4326')

        # Criar o mapa
        centro = gdf_sel.geometry.centroid.iloc[0]
        mapa = folium.Map(location=[centro.y, centro.x], zoom_start=10)

        # Adiciona bairro selecionado em azul com tooltip (formato original)
        folium.GeoJson(gdf_sel,
                       name="Bairro selecionado",
                       style_function=lambda x: {"fillColor": "blue", "color": "blue", "weight": 2, "fillOpacity": 0.4},
                       tooltip=folium.GeoJsonTooltip(fields=['nome'], aliases=['Bairro:'])
                      ).add_to(mapa)

        # Adiciona vizinhos em vermelho com tooltip
        folium.GeoJson(gdf_viz,
                       name="Vizinhos",
                       style_function=lambda x: {"fillColor": "red", "color": "red", "weight": 1.5, "fillOpacity": 0.3},
                       tooltip=folium.GeoJsonTooltip(fields=['nome'], aliases=['Bairros:'])
                      ).add_to(mapa)

        folium.LayerControl().add_to(mapa)
        display(mapa)

# 4. Conectar a função ao Dropdown
comboMun.observe(atualizar_mapa)

# 5. Mostrar o Dropdown
display(comboMun)

In [ ]:
!sudo shp2pgsql -c -I -s 4326 -W "latin1" /content/drive/MyDrive/Dados/ESCOLA_MUNICIPAL.shp public.escolas1 | psql -d postgres -U root /dev/null 2>&1

df_escolas = %sql SELECT * FROM escolas1
df_escolas.head()

In [ ]:
!sudo shp2pgsql -c -I -s 4326 -W "latin1" /content/drive/MyDrive/Dados/CMEI.shp public.cmeis1 | psql -d postgres -U root > /dev/null 2>&1

df_cmeis = %sql SELECT * FROM cmeis1
df_cmeis.head()

In [ ]:
!sudo shp2pgsql -c -I -s 4326 -W "latin1" /content/drive/MyDrive/Dados/CEI_CONVENIADA.shp public.ceis1 | psql -d postgres -U root > /dev/null 2>&1

df_ceis = %sql SELECT * FROM ceis1
df_ceis.head()

In [ ]:
%%sql
DROP TABLE IF EXISTS disp_educacionais CASCADE;

CREATE TABLE disp_educacionais AS
SELECT *
FROM escolas1
UNION ALL
SELECT *
FROM cmeis1
UNION ALL
SELECT *
FROM ceis1;

In [ ]:
df_todas_instituicoes = %sql SELECT * FROM disp_educacionais;
df_todas_instituicoes.head()

In [ ]:
import geopandas as gpd
import folium
from shapely import wkt
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.pyplot as plt # Adicionado para a nova forma de obter colormaps

# Consulta todas as instituições educacionais
consulta_todas_instituicoes = """
    SELECT gid, equipament, tipo_equi, lat_sirgas, lon_sirgas
    FROM disp_educacionais
"""
df_all_instituicoes = %sql $consulta_todas_instituicoes

if not df_all_instituicoes.empty:
    # Cria a coluna de geometria a partir de lat_sirgas e lon_sirgas
    gdf_all_instituicoes = gpd.GeoDataFrame(
        df_all_instituicoes,
        geometry=gpd.points_from_xy(df_all_instituicoes['lon_sirgas'], df_all_instituicoes['lat_sirgas']),
        crs='EPSG:4326'
    )

    # Cria um mapa de cores para 'tipo_equi'
    unique_tipos = gdf_all_instituicoes['tipo_equi'].unique()
    num_colors = len(unique_tipos)

    # Usar matplotlib.colormaps.get_cmap como recomendado para evitar o aviso de depreciação
    cmap_name = 'tab10' if num_colors <= 10 else 'gist_rainbow' # Escolhe o colormap
    cmap = plt.colormaps.get_cmap(cmap_name) # Obtém o objeto colormap

    color_map = {}
    if num_colors > 0:
        for i, tipo in enumerate(unique_tipos):
            # Normaliza o índice para o colormap, tratando o caso de um único tipo
            normalized_i = i / (num_colors - 1) if num_colors > 1 else 0.0
            color_map[tipo] = colors.to_hex(cmap(normalized_i))

    # Calcula o centro e o zoom para o mapa baseado em todas as instituições
    minx, miny, maxx, maxy = gdf_all_instituicoes.total_bounds
    centro_lat = (miny + maxy) / 2
    centro_lon = (minx + maxx) / 2

    # Ajusta o zoom para cobrir todas as instituições
    mapa_nao_interativo = folium.Map(location=[centro_lat, centro_lon], zoom_start=11)

    # Adiciona todas as instituições ao mapa iterando e criando CircleMarkers
    for idx, row in gdf_all_instituicoes.iterrows():
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=5,
            fill_color=color_map[row['tipo_equi']],
            color='black',
            weight=1,
            fill_opacity=0.7,
            tooltip=f"Instituição: {row['equipament']}<br>Tipo: {row['tipo_equi']}"
        ).add_to(mapa_nao_interativo)

    # Adiciona uma legenda para os tipos de instituição
    if color_map:
        legend_html = '<div style="position: fixed; bottom: 50px; left: 50px; width: 200px; background-color: white; border:2px solid grey; z-index:9999; font-size:14px; padding: 10px;">'
        legend_html += '<b>Tipo de Instituição</b><br>'
        for tipo, color in color_map.items():
            legend_html += f'<i style="background:{color}; width:10px; height:10px; display:inline-block; margin-right:5px;"></i>{tipo}<br>'
        legend_html += '</div>'
        mapa_nao_interativo.get_root().html.add_child(folium.Element(legend_html))

    folium.LayerControl().add_to(mapa_nao_interativo)
    display(mapa_nao_interativo)
else:
    print("Nenhuma instituição educacional encontrada na tabela 'disp_educacionais'.")


In [ ]:
from ipywidgets import Dropdown
from IPython.display import display, clear_output
import pandas as pd
import geopandas as gpd
import folium
from shapely import wkt
import matplotlib.cm as cm
import matplotlib.colors as colors

# 1. Buscar bairros
df_bairros = %sql SELECT codigo, nome FROM bairros1 ORDER BY nome

# 2. Criar Dropdown
comboMun = Dropdown(
    options=[(row['nome'], row['codigo']) for _, row in df_bairros.iterrows()],
    description='Bairros:'
)

# 3. Função para atualizar o mapa
def atualizar_mapa(change):
    if change['type'] == 'change' and change['name'] == 'value':
        clear_output(wait=True)
        display(comboMun)

        codigo = comboMun.value
        selected_bairro_name = comboMun.label # Get the name of the selected bairro

        # Consulta bairro selecionado (original shape)
        consulta_sel = f"""
            SELECT codigo, nome, ST_AsText(ST_Transform(geom, 4326)) AS geom
            FROM bairros1
            WHERE codigo = '{codigo}'
        """
        df_sel = %sql $consulta_sel
        df_sel['geometry'] = df_sel['geom'].apply(wkt.loads)
        gdf_sel = gpd.GeoDataFrame(df_sel, geometry='geometry', crs='EPSG:4326')

        # Consulta todas as instituições educacionais dentro do bairro selecionado
        consulta_todas_instituicoes = f"""
            SELECT gid, equipament, tipo_equi, lat_sirgas, lon_sirgas
            FROM disp_educacionais d, bairros1 b
            WHERE b.codigo = {codigo}
              AND ST_Within(ST_SetSRID(ST_MakePoint(d.lon_sirgas, d.lat_sirgas), 4326), b.geom)
        """
        df_all_instituicoes = %sql $consulta_todas_instituicoes

        if not df_all_instituicoes.empty:
            # Create GeoDataFrame from lat_sirgas and lon_sirgas
            gdf_all_instituicoes = gpd.GeoDataFrame(
                df_all_instituicoes,
                geometry=gpd.points_from_xy(df_all_instituicoes['lon_sirgas'], df_all_instituicoes['lat_sirgas']),
                crs='EPSG:4326'
            )

            # Create a color map for 'tipo_equi'
            unique_tipos = gdf_all_instituicoes['tipo_equi'].unique()
            num_colors = len(unique_tipos)
            # Use a categorical colormap for better distinction
            cmap = cm.get_cmap('tab10' if num_colors <= 10 else 'gist_rainbow', num_colors)
            color_map = {tipo: colors.to_hex(cmap(i)) for i, tipo in enumerate(unique_tipos)}
        else:
            # Create an empty GeoDataFrame consistent with the selected columns
            gdf_all_instituicoes = gpd.GeoDataFrame(columns=[
                'gid', 'equipament', 'tipo_equi', 'lat_sirgas', 'lon_sirgas', 'geometry'
            ], geometry='geometry', crs='EPSG:4326')
            color_map = {} # Empty color map if no institutions

        # Criar o mapa
        centro = gdf_sel.geometry.centroid.iloc[0]
        mapa = folium.Map(location=[centro.y, centro.x], zoom_start=12) # Aumenta o zoom para ver as instituições

        # Adiciona bairro selecionado em azul com tooltip (formato original)
        folium.GeoJson(gdf_sel,
                       name="Bairro selecionado",
                       style_function=lambda x: {"fillColor": "blue", "color": "blue", "weight": 2, "fillOpacity": 0.4},
                       tooltip=folium.GeoJsonTooltip(fields=['nome'], aliases=['Bairro:'])
                      ).add_to(mapa)

        # Adiciona instituições educacionais usando CircleMarkers
        if not gdf_all_instituicoes.empty:
            for idx, row in gdf_all_instituicoes.iterrows():
                folium.CircleMarker(
                    location=[row.geometry.y, row.geometry.x],
                    radius=5,
                    fill_color=color_map[row['tipo_equi']],
                    color='black', # Border color
                    weight=1,
                    fill_opacity=0.7,
                    tooltip=f"Instituição: {row['equipament']}<br>Tipo: {row['tipo_equi']}"
                ).add_to(mapa)

        # Adiciona legenda bom base nos tipos de instituições
        if color_map:
            legend_html = '<div style="position: fixed; bottom: 50px; left: 50px; width: 200px; background-color: white; border:2px solid grey; z-index:9999; font-size:14px; padding: 10px;">'
            legend_html += '<b>Tipo de Instituição</b><br>'
            for tipo, color in color_map.items():
                legend_html += f'<i style="background:{color}; width:10px; height:10px; display:inline-block; margin-right:5px;"></i>{tipo}<br>'
            legend_html += '</div>'
            mapa.get_root().html.add_child(folium.Element(legend_html))

        folium.LayerControl().add_to(mapa)
        display(mapa)

# 4. Conectar a função ao Dropdown
comboMun.observe(atualizar_mapa)

# 5. Mostrar o Dropdown
display(comboMun)

In [ ]:
import folium
import geopandas as gpd
from shapely import wkt

# Reload the sql extension to try and clear sticky configs
%reload_ext sql
%config SqlMagic.feedback=False
%config SqlMagic.autopandas=True  # Ensure autopandas is enabled
%sql postgresql+psycopg2://@/postgres

# Solicita as coordenadas ao usuário
try:
    input_lat = float(input("Digite a latitude (ex: -25.43): "))
    input_lon = float(input("Digite a longitude (ex: -49.27): "))
except ValueError:
    print("Entrada inválida. Por favor, digite números para latitude e longitude.")
    exit()

# Define a sessão PostGIS
%sql SET SESSION search_path TO public;

# Consulta o bairro que contém o ponto inserido
# ST_MakePoint(longitude, latitude) - atenção à ordem
consulta_bairro = """
    SELECT nome, ST_AsText(geom) AS geometria
    FROM bairros1
    WHERE ST_Within(ST_SetSRID(ST_MakePoint(:input_lon, :input_lat), 4326), geom);
"""

# Execute the SQL query. With autopandas=True, this should directly return a DataFrame.
df_bairro_encontrado = %sql $consulta_bairro

# Cria o mapa Folium centrado no ponto inserido
mapa = folium.Map(location=[input_lat, input_lon], zoom_start=13)

# Adiciona o ponto inserido ao mapa
folium.Marker(
    location=[input_lat, input_lon],
    tooltip=f"Coordenadas: ({input_lat}, {input_lon})",
    icon=folium.Icon(color='green')
).add_to(mapa)

if not df_bairro_encontrado.empty:
    bairro_nome = df_bairro_encontrado['nome'].iloc[0]
    bairro_wkt = df_bairro_encontrado['geometria'].iloc[0]

    # Converte WKT para GeoDataFrame para adicionar ao Folium
    gdf_bairro = gpd.GeoDataFrame(
        {'nome': [bairro_nome]},
        geometry=[wkt.loads(bairro_wkt)],
        crs='EPSG:4326'
    )

    # Adiciona o polígono do bairro ao mapa
    folium.GeoJson(
        gdf_bairro,
        name=f"Bairro: {bairro_nome}",
        style_function=lambda x: {"fillColor": "orange", "color": "red", "weight": 2, "fillOpacity": 0.4},
        tooltip=folium.GeoJsonTooltip(fields=['nome'], aliases=['Bairro:'])
    ).add_to(mapa)

    print(f"O ponto ({input_lat}, {input_lon}) está dentro do bairro: {bairro_nome}")
else:
    print(f"O ponto ({input_lat}, {input_lon}) não foi encontrado em nenhum bairro registrado.")

# Exibe o mapa
folium.LayerControl().add_to(mapa)
display(mapa)